# Vector Search with DocumentDB

This notebook implements semantic search using pre-embedded Airbnb listings data.

## Learning Objectives
- Connect to DocumentDB with pre-embedded data
- Create vector search indexes using cosmosSearch
- Implement semantic search with similarity scoring
- Apply filters to refine search results

**💡 Embedding Strategy:** The `descriptionVector` field is generated from a **composite text** that combines `name`, `property_type`, cleaned `description`, `neighborhood_overview`, and top amenities — not just the raw description. This improves search relevance by giving the embedding model richer context. See [generate-embeddings.ipynb](./generate-embeddings.ipynb) for details.

**📚 Prerequisites:** If you want to understand how embeddings are generated, see [generate-embeddings.ipynb](./generate-embeddings.ipynb) first.

## Step 1: Setup and Connect to DocumentDB

The `embedded_data.json` file contains all 1,000 listings with their `descriptionVector` fields already populated using composite text (name + cleaned description + neighborhood + property type + amenities).

In [ ]:
import os
import json
from pymongo import MongoClient
from openai import OpenAI
from dotenv import load_dotenv

# Load environment variables
load_dotenv(override=True)

# Initialize OpenAI client (needed for query embeddings)
openai_client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))

# Connect to DocumentDB
DOCUMENTDB_CONNECTION_STRING = os.getenv('DOCUMENTDB_CONNECTION_STRING')
client = MongoClient(DOCUMENTDB_CONNECTION_STRING)
db = client['db']
collection = db['listings']

print("✅ Connected to DocumentDB")
print(f"📊 Current document count: {collection.count_documents({})}")

# Verify connection by fetching one document
test_doc = collection.find_one()
if test_doc:
    print(f"✅ Successfully retrieved document: {test_doc.get('name', 'Unknown')}")
else:
    print("⚠️ No documents found. Load embedded_data.json first.")

## Step 2: Create Vector Index Using the DocumentDB for VS Code Extension

You need to create a **vector search index** to enable fast similarity searches.

1. **Open the DocumentDB Extension** in VS Code (click the database icon in the sidebar)

2. **Navigate to your Scrapbook**:
   - Right-click on your connection
   - Select **"New Scrapbook"**

3. **Run the following commands** in your scrapbook (select each block and press `Ctrl+Enter` or click "Run"):

```json
// Create vector search index on the descriptionVector field
db.runCommand({
    createIndexes: "listings",
    indexes: [{
        key: { "descriptionVector": "cosmosSearch" },
        name: "vectorSearchIndex",
        cosmosSearchOptions: {
            kind: "vector-ivf",
            numLists: 100,
            similarity: "COS",
            dimensions: 1536
        }
    }]
})

// Check all indexes on the collection
db.listings.getIndexes()
```

### 💡 Understanding Index Parameters

DocumentDB supports native vector search using the `cosmosSearch` operator. We'll use **IVF (Inverted File Index)** for efficient approximate search.

| Parameter | Value | Description |
|-----------|-------|-------------|
| `kind` | `"vector-ivf"` | Uses Inverted File Index for fast approximate search |
| `numLists` | `100` | Number of clusters (higher = more accurate but slower) |
| `similarity` | `"COS"` | Cosine similarity (range: 0 to 1, where 1 = identical) |
| `dimensions` | `1536` | Must match embedding size (OpenAI text-embedding-3-small) |

## Step 3: Create Embedding Function for Queries

We need to convert search queries into embeddings to compare against stored vectors.

In [ ]:
def generate_embedding(text):
    """
    Generate a vector embedding for the given text using OpenAI.
    
    Args:
        text (str): The text to embed
        
    Returns:
        list: A 1536-dimension vector representing the text
    """
    if not text or not isinstance(text, str):
        return None
    
    try:
        response = openai_client.embeddings.create(
            model="text-embedding-3-small",
            input=text
        )
        return response.data[0].embedding
    except Exception as e:
        print(f"Error generating embedding: {e}")
        return None

# Test the function
test_embedding = generate_embedding("cozy apartment downtown")
print(f"✅ Embedding function ready (vector size: {len(test_embedding)})")

## Step 4: Implement Basic Semantic Search

Now we can search for listings using natural language! The search converts the query to an embedding and finds the most similar listings.

### Understanding Search Scores:
- Scores range from 0 to 1 (with cosine similarity)
- **> 0.75**: Strong semantic relevance
- **0.5 - 0.75**: Moderately relevant
- **< 0.5**: Weak matches

In [ ]:
def search_listings(query, limit=5):
    """
    Search for listings using semantic similarity.
    
    Args:
        query (str): Natural language search query
        limit (int): Maximum number of results to return
        
    Returns:
        list: Matching listings with similarity scores
    """
    # Generate embedding for the query
    query_embedding = generate_embedding(query)
    
    if not query_embedding:
        print("❌ Failed to generate query embedding")
        return []
    
    # Perform vector search using cosmosSearch
    pipeline = [
        {
            "$search": {
                "cosmosSearch": {
                    "vector": query_embedding,
                    "path": "descriptionVector",
                    "k": limit  # Number of nearest neighbors
                },
                "returnStoredSource": True
            }
        },
        {
            "$project": {
                "_id": 1,
                "name": 1,
                "description": 1,
                "property_type": 1,
                "bedrooms": 1,
                "beds": 1,
                "price": 1,
                "address.market": 1,
                "amenities": 1,
                "searchScore": {"$meta": "searchScore"}
            }
        }
    ]
    
    results = list(collection.aggregate(pipeline))
    return results

In [ ]:
# Test the search
query = "cozy apartment with parking near downtown"
results = search_listings(query, limit=5)

print(f"\n🔍 Search Query: '{query}'")
print(f"📊 Found {len(results)} results\n")

for idx, result in enumerate(results, 1):
    print(f"{idx}. {result['name']}")
    print(f"   Property Type: {result.get('property_type', 'N/A')}")
    print(f"   Location: {result.get('address', {}).get('market', 'N/A')}")
    print(f"   Bedrooms: {result.get('bedrooms', 'N/A')} | Price: {result.get('price', 'N/A')}")
    print(f"   Similarity Score: {result.get('searchScore', 0):.4f}")
    print(f"   Preview: {result.get('description', '')[:100]}...")
    print()

## Step 5: Search with Filters

Combine semantic search with traditional filters like bedrooms, price, market, and amenities for more refined results.

In [ ]:
def search_listings_with_filters(query, filters=None, limit=5):
    """
    Search for listings with semantic similarity and additional filters.
    
    Args:
        query (str): Natural language search query
        filters (dict): Optional filters (bedrooms, price_max, market, amenities)
        limit (int): Maximum number of results to return
        
    Returns:
        list: Matching listings with similarity scores
    """
    # Generate embedding for the query
    query_embedding = generate_embedding(query)
    
    if not query_embedding:
        print("❌ Failed to generate query embedding")
        return []
    
    # Build match stage for filters
    match_conditions = {}
    
    if filters:
        if 'bedrooms' in filters:
            match_conditions['bedrooms'] = {"$gte": filters['bedrooms']}
        
        if 'price_max' in filters:
            match_conditions['price'] = {"$lte": filters['price_max']}
        
        if 'market' in filters:
            match_conditions['address.market'] = filters['market']
        
        if 'amenities' in filters:
            # Amenities is a list, so we check if all required amenities are present
            match_conditions['amenities'] = {"$all": filters['amenities']}
    
    # Build aggregation pipeline
    pipeline = [
        {
            "$search": {
                "cosmosSearch": {
                    "vector": query_embedding,
                    "path": "descriptionVector",
                    "k": limit * 10  # Fetch more to account for filtering
                },
                "returnStoredSource": True
            }
        }
    ]
    
    # Add filter stage if we have conditions
    if match_conditions:
        pipeline.append({"$match": match_conditions})
    
    # Add projection and limit
    pipeline.extend([
        {
            "$project": {
                "_id": 1,
                "name": 1,
                "description": 1,
                "property_type": 1,
                "bedrooms": 1,
                "beds": 1,
                "price": 1,
                "address.market": 1,
                "amenities": 1,
                "searchScore": {"$meta": "searchScore"}
            }
        },
        {"$limit": limit}
    ])
    
    results = list(collection.aggregate(pipeline))
    return results

In [ ]:
# Test with filters
query = "family-friendly home with outdoor space"
filters = {
    "bedrooms": 3,
    "price_max": 200,
    "amenities": ["Wifi", "Kitchen"]
}

results = search_listings_with_filters(query, filters, limit=5)

print(f"\n🔍 Search Query: '{query}'")
print(f"🎯 Filters:")
print(f"   - Bedrooms: {filters['bedrooms']}+")
print(f"   - Max Price: {filters['price_max']}")
print(f"   - Amenities: {', '.join(filters['amenities'])}")
print(f"\n📊 Found {len(results)} results\n")

for idx, result in enumerate(results, 1):
    print(f"{idx}. {result['name']}")
    print(f"   Property Type: {result.get('property_type', 'N/A')}")
    print(f"   Location: {result.get('address', {}).get('market', 'N/A')}")
    print(f"   Bedrooms: {result.get('bedrooms', 'N/A')} | Price: {result.get('price', 'N/A')}")
    print(f"   Similarity Score: {result.get('searchScore', 0):.4f}")
    amenities_preview = ', '.join(result.get('amenities', [])[:5])
    print(f"   Amenities: {amenities_preview}...")
    print()

## Step 6: Experiment with Different Queries

Test the semantic search with various natural language queries to see how it understands context and intent.

**💡 Observations:**
- "romantic getaway" finds properties with ambiance descriptions
- "pet-friendly" matches listings mentioning pets, animals, or outdoor areas
- "business travel" finds properties with workspaces and good wifi
- The semantic understanding goes beyond exact keyword matching!

In [ ]:
# Test various semantic queries
test_queries = [
    "romantic getaway for couples",
    "pet-friendly place near parks",
    "business travel with home office",
    "beachfront property for surfing",
    "quiet retreat for meditation and yoga"
]

print("🧪 Testing Semantic Search Capabilities\n")
print("=" * 80)

for query in test_queries:
    results = search_listings(query, limit=3)
    
    print(f"\n🔍 Query: '{query}'")
    print(f"📊 Top 3 Results:")
    
    for idx, result in enumerate(results, 1):
        print(f"\n   {idx}. {result['name']}")
        print(f"      Score: {result.get('searchScore', 0):.4f}")
        print(f"      {result.get('property_type', 'N/A')} | "
              f"{result.get('bedrooms', 'N/A')} bed | "
              f"{result.get('price', 'N/A')}/night")
    
    print("\n" + "-" * 80)

## 🎉 Next Steps

Now that you understand vector search, continue to **[Module 1](../exercises/Module-01.md#launch-the-backend-terminal-1)** to run the application and see your chamges live!